# 1 – Simple RAG

The baseline that every other example builds on. A retrieval-augmented generation (RAG) pipeline has five steps:

| Step | Name | What happens |
|---|---|---|
| 1 | **LOAD** | read raw documents from disk |
| 2 | **CHUNK** | split them into small pieces (recursive splitter) |
| 3 | **EMBED** | turn every chunk into a vector |
| 4 | **STORE** | put the vectors into a vector database (Chroma) |
| 5 | **RETRIEVE + GENERATE** | embed the question, find the most similar chunks, and let an LLM answer using only those chunks |

Steps 1–4 are the *indexing phase*: in a real system they run once, offline. Step 5 is the *query phase*: it runs for every user question.

**Before you run this notebook**

- Install the dependencies from `requirements.txt` (see the README) and start Jupyter from the repository folder, so that `data/` sits next to this notebook.
- Have [Ollama](https://ollama.com) running and pull the two models used here: `ollama pull mistral-small3.2:24b` (the LLM) and `ollama pull embeddinggemma:300m` (the embedding model). Both run locally; no API key is needed. Set the `RAG_LLM` environment variable in the shell that launches Jupyter to use a different chat model.

## Imports

Everything comes from the LangChain ecosystem except the standard library:

- `init_chat_model` – one factory function for any chat model. You pass a `"provider:model"` string and it instantiates the right class (`ChatOllama`, `ChatOpenAI`, `ChatAnthropic`, ...).
- `Chroma` – LangChain's wrapper around the Chroma vector database.
- `Document` – LangChain's unit of text: `page_content` (a string) plus `metadata` (a dict).
- `ChatPromptTemplate`, `StrOutputParser`, `RunnableLambda`, `RunnablePassthrough` – building blocks of the LangChain Expression Language (LCEL) that we chain together in step 5.
- `OllamaEmbeddings` – embeds text with a model served by the local Ollama instance (`langchain-ollama`).
- `RecursiveCharacterTextSplitter` – the chunker used in step 2.

In [ ]:
import os
from pathlib import Path

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Configuration

- `DATA_DIR` – the folder with the three fictional documents (`data/`, next to this notebook).
- `LLM_MODEL` – any model supported by `init_chat_model`, read from the `RAG_LLM` environment variable. The default is `"ollama:mistral-small3.2:24b"`, served by the local Ollama instance; `"ollama:<any model you pulled>"` works the same way, and hosted providers such as `"openai:gpt-4o-mini"` need their API key and integration package.
- `EMBEDDING_MODEL` – `embeddinggemma:300m`, Google's 300M-parameter embedding model (768 dimensions), also served by Ollama. Small (~620 MB) and fast.
- `CHUNK_SIZE` – maximum number of characters per chunk.
- `CHUNK_OVERLAP` – characters shared between neighbouring chunks, so a sentence cut at a boundary is still complete in one of them.
- `TOP_K` – how many chunks we hand to the LLM.

In [ ]:
DATA_DIR = Path("data")

LLM_MODEL = os.getenv("RAG_LLM", "ollama:mistral-small3.2:24b")

EMBEDDING_MODEL = "embeddinggemma:300m"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 4

print("LLM:", LLM_MODEL)

## Step 1 – LOAD

A LangChain `Document` is just `page_content` (the text) plus `metadata` (a dict). We read every `.txt` file in `data/` into one `Document` and put the file name into `metadata["source"]`; later we use it to show where an answer came from.

For PDFs, HTML, Word files and so on, LangChain has document loaders (for example `PyPDFLoader` in `langchain-community`). They all return the same `Document` objects, so the rest of the pipeline does not change.

In [ ]:
docs = [
    Document(page_content=path.read_text(encoding="utf-8"),
             metadata={"source": path.name})
    for path in sorted(DATA_DIR.glob("*.txt"))
]
print(f"Loaded {len(docs)} documents")
for doc in docs:
    print(f"  {doc.metadata['source']}: {len(doc.page_content)} characters")

## Step 2 – CHUNK

Why chunk at all?

- Embedding models have a maximum input length.
- One vector for a whole report is a blurry average of many topics; one vector per paragraph is sharp.
- We only want to send the *relevant* bits to the LLM, not the whole corpus.

`RecursiveCharacterTextSplitter` is the default, structure-agnostic chunker. It tries to split on the largest natural boundary first and only falls back to a smaller one when a piece is still too big. The `separators` are tried in this order:

1. `"\n\n"` – paragraph
2. `"\n"` – line
3. `r"(?<=\. )"` – a regex *lookbehind* that splits right **after** a full stop followed by a space, so sentences stay whole and keep their period
4. `" "` – word
5. `""` – character (last resort)

Because one of the separators is a regular expression we set `is_separator_regex=True`. `add_start_index=True` stores the character offset of every chunk inside its document in `metadata["start_index"]` (Example 4 depends on that).

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    is_separator_regex=True,
    add_start_index=True,
)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

Let's look at one chunk. The metadata was copied from the parent document and `start_index` was added by the splitter. Notice that the text never says *which* company "the company" is – Examples 2 and 4 deal with exactly this problem.

In [ ]:
chunk = next(c for c in chunks if "revenue rose" in c.page_content)
print(chunk.metadata)
print(chunk.page_content)

## Step 3 – EMBED

The embedding model maps text to a vector such that texts with similar meaning end up close together. `OllamaEmbeddings` sends each text to the local Ollama server, which runs the model and returns the vector. Ollama's embedding endpoint returns unit-length vectors, so the inner product of two vectors equals their cosine similarity (1.0 = identical direction).

Let's embed one sentence to see what a vector looks like: 768 floats.

In [ ]:
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

vector = embeddings.embed_query("What was the revenue in 2025?")
print(len(vector), "dimensions, first five values:", [round(v, 3) for v in vector[:5]])

## Step 4 – STORE

Chroma is an open-source vector database. Here it runs in memory; pass `persist_directory="./chroma_db"` to keep the index on disk between runs. You could swap it for pgvector, Qdrant, Pinecone, ... – the LangChain interface (`add_documents`, `similarity_search`, `as_retriever`) stays the same.

- `collection_metadata={"hnsw:space": "cosine"}` tells Chroma to compare vectors by cosine distance. HNSW is the approximate-nearest-neighbour index Chroma builds internally, so search stays fast even with millions of vectors.
- `reset_collection()` empties the collection before we fill it. Chroma keeps its in-memory collections alive for the whole kernel session, so without this line re-running the cell would add every chunk a second time. (In a script you would simply write `Chroma.from_documents(chunks, embeddings, ...)`.)
- `add_documents` embeds every chunk with the model from step 3 and stores text, metadata and vector together.

In [ ]:
vector_store = Chroma(
    collection_name="simple_rag",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)
vector_store.reset_collection()
ids = vector_store.add_documents(chunks)
print(f"Indexed {len(ids)} chunks")

## Step 5a – RETRIEVE (similarity search)

The question is embedded with the **same** model, and Chroma returns the `TOP_K` chunk vectors closest to the question vector. `similarity_search_with_relevance_scores` converts Chroma's distance into a similarity:

```
score = 1 - cosine_distance = cosine similarity   (higher = better)
```

In [ ]:
def show_retrieval(vector_store, question):
    print(f"\nTop {TOP_K} chunks for: {question!r}")
    results = vector_store.similarity_search_with_relevance_scores(question, k=TOP_K)
    for rank, (doc, score) in enumerate(results, start=1):
        source = Path(doc.metadata["source"]).name
        preview = doc.page_content[:90].replace("\n", " ")
        print(f"  {rank}. score={score:.3f}  [{source}]  {preview}...")


show_retrieval(vector_store, "What was Novaterra's revenue in 2025?")

## Step 5b – GENERATE

The prompt *grounds* the LLM: answer only from the context, say so if the answer is not there, and cite which chunks were used. `{context}` and `{question}` are template variables that are filled in at run time.

`format_docs` turns the retrieved `Document`s into one numbered block of text, so the citations `[1]`, `[2]` in the answer can be traced back to a chunk and its source file.

In [ ]:
PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions using ONLY the context below. "
     "If the context does not contain the answer, say you don't know. "
     "Cite the chunks you used like [1], [2].\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])


def format_docs(docs):
    return "\n\n".join(
        f"[{i}] (source: {Path(d.metadata['source']).name})\n{d.page_content}"
        for i, d in enumerate(docs, start=1)
    )

### Wiring it together with LCEL

- `vector_store.as_retriever(...)` returns a *retriever*: a Runnable that maps a question (`str`) to a `list[Document]`.
- `init_chat_model(LLM_MODEL, temperature=0)` creates the chat model; `temperature=0` makes the answers as deterministic as possible.
- In the LangChain Expression Language (LCEL) the `|` operator pipes the output of one step into the next. A dict of Runnables runs its values on the same input and collects the results under the given keys:

```
question ──┬─> retriever -> format_docs ──> {context}
           └─> RunnablePassthrough ──────> {question}
                                               │
                        PROMPT -> LLM -> StrOutputParser -> plain string
```

`RunnablePassthrough()` forwards the input unchanged, `RunnableLambda(format_docs)` turns a plain Python function into a Runnable, and `StrOutputParser()` extracts the text from the model's message object.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})
llm = init_chat_model(LLM_MODEL, temperature=0)

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | PROMPT
    | llm
    | StrOutputParser()
)

## Run it

The index is built; every question now only needs the query phase. For each question we first show which chunks were retrieved, then the grounded answer. Edit `QUESTIONS` to ask your own.

In [ ]:
QUESTIONS = [
    "What was Novaterra's revenue in 2025?",
    "Which credit score leads to an automatic decline at Meridian Bank?",
    "How often do the Harbor City ferries run in winter?",
]

for question in QUESTIONS:
    show_retrieval(vector_store, question)
    print("\nAnswer:", rag_chain.invoke(question))
    print("-" * 80)

## Things to try

1. Ask your own question in the cell below.
2. Change `CHUNK_SIZE` to 200 and then to 1500 and re-run the cells from *Step 2* onward. Which answers get worse, and why?
3. A few headings are still stuck to the end of the previous chunk. How would a structure-aware splitter (`MarkdownHeaderTextSplitter`) handle that?

In [ ]:
print(rag_chain.invoke("What is the maximum term of a personal loan?"))